In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import igraph as ig
from networkx.algorithms import bipartite



color_map_cat = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black', 'ROE': 'orange'}
color_map_cat_no_roe = {'ASN': 'red', 'NGO': 'yellow', 'GOV': 'green', 'OTHER': 'blue', 'PEC': 'black'}

In [ ]:
# Load the data
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')
h_nodes = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='nodes')
h_edges = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='edges')

In [ ]:
tw_nodes

In [ ]:
tw_nodes['Category'].unique()

In [ ]:
h_nodes

In [ ]:
seeds_nodes = h_nodes[h_nodes['STATUS']=='IN'].copy()
seeds_nodes.loc[:, 'Stance'] = [0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]

In [ ]:
h_nodes.shape

In [ ]:
h_edges

# Hyperlink

In [ ]:
h_nodes = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='nodes')
h_edges = pd.read_excel('Hyperlink_data_def.xlsx', sheet_name='edges')
h_edges = h_edges[h_edges['Source'] != h_edges['Target']]
h_edges = h_edges.groupby(['Source', 'Target']).size().reset_index(name='weight')
h_nodes.loc[27, 'Label'] = 'European Union'
# in h_nodes['STATUS'] change 'IN' to Seed and 'DISCOVERED' to 'Disc'
h_nodes['STATUS'] = h_nodes['STATUS'].replace({'IN': 'Seed', 'DISCOVERED': 'Disc'})
seeds_nodes = h_nodes[h_nodes['STATUS']=='Seed'].copy()
seeds_nodes.loc[:, 'Stance'] = [0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]

In [ ]:
# create a directed graph among only nodes with 'STATUS' == 'IN'
h_net_seeds = nx.DiGraph()
h_net_sd = nx.DiGraph()
for i in range(h_nodes.shape[0]):
    h_net_sd.add_node(h_nodes['ID'][i], status=h_nodes['STATUS'][i], label=h_nodes['Label'][i], category=h_nodes['Category'][i])
    if h_nodes['STATUS'][i] == 'Seed':
        h_net_seeds.add_node(h_nodes['ID'][i], status=h_nodes['STATUS'][i], label=h_nodes['Label'][i], category=h_nodes['Category'][i])
for i in range(h_edges.shape[0]):
    if h_edges['Source'][i] in h_net_seeds.nodes() and h_edges['Target'][i] in h_net_seeds.nodes():
        h_net_seeds.add_edge(h_edges['Source'][i], h_edges['Target'][i])
    else:
        h_net_sd.add_edge(h_edges['Source'][i], h_edges['Target'][i])

# remove self loops
h_net_seeds.remove_edges_from(nx.selfloop_edges(h_net_seeds))
#h_net_sd.remove_edges_from(nx.selfloop_edges(h_net_sd))



In [ ]:
h_net_seeds.number_of_nodes(), h_net_seeds.number_of_edges()

In [ ]:
h_net_sd.number_of_nodes(), h_net_sd.number_of_edges()

In [ ]:
from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

# plot the network with edge width based on the counts of the edges
plt.figure(figsize=(12, 8))

# reuse existing layout/sizes/colors if available
pos = pos if 'pos' in globals() else nx.spring_layout(G, k=0.5, iterations=6)
node_sizes = node_sizes if 'node_sizes' in globals() else [(G.nodes[n]['Seeds'] + 1) * 1000 for n in G.nodes()]
node_colors = node_colors if 'node_colors' in globals() else [color_map_cat.get(n, 'gray') for n in G.nodes()]

nx.draw_networkx_nodes(
    G, pos,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.8,
    edgecolors='black',   # visible outline around nodes
    linewidths=2.0
)

edge_widths = [max(1.5, G[u][v]['counts'] / 4) for u, v in G.edges()]
arrows = nx.draw_networkx_edges(
    G, pos,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,
    connectionstyle='arc3,rad=0.12',
    min_source_margin=30,   # keep arrows off node centers
    min_target_margin=33
)

# add a white outline to arrows for better definition
try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

# no labels on nodes
# nx.draw_networkx_labels(G, pos, font_size=20, font_weight='bold')

# legend with node names
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(n, 'gray'),
           markeredgecolor='black', label=n)
    for n in G.nodes()
]
plt.legend(handles=legend_elements, title='Category',
           loc='center left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.axis('off')
plt.tight_layout()
# plt.savefig('inter_category_network.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

# plot h_net_seeds
plt.figure(figsize=(12, 8))

#pos_h = nx.spring_layout(h_net_seeds, k=0.5, iterations=50)
pos_h = pos_h

node_colors = [color_map_cat.get(h_net_seeds.nodes[n]['category'], 'gray') for n in h_net_seeds.nodes()]
node_sizes = [450 for _ in h_net_seeds.nodes()]

nx.draw_networkx_nodes(
    h_net_seeds, pos_h,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.6,
    edgecolors='black',
    linewidths=2.0
)

edge_widths = [1.8 for _ in h_net_seeds.edges()]
arrows = nx.draw_networkx_edges(
    h_net_seeds, pos_h,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,
    connectionstyle='arc3,rad=0.12',
    min_source_margin=2,
    min_target_margin=2
)

# Add node ID labels
nx.draw_networkx_labels(
    h_net_seeds, pos_h,
    labels={n: n for n in h_net_seeds.nodes()},
    font_size=12,
    font_weight='bold'
)

try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

uniq_cats = sorted({h_net_seeds.nodes[n]['category'] for n in h_net_seeds.nodes()})
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats
]
plt.legend(handles=legend_elements, title='Category',
           loc='upper left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Seed Hyperlink Network')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# compute the degree of each node in h_net_seeds
h_net_seeds_indegree = dict(h_net_seeds.in_degree())

# compute the out-degree of each node in h_net_seeds
h_net_seeds_outdegree = dict(h_net_seeds.out_degree())

# histogram of in-degrees
plt.hist(list(h_net_seeds_indegree.values()), bins=9, color='red', alpha=0.7)
plt.xlabel('In-degree')
plt.ylabel('Frequency')
plt.title('Histogram of in-degrees')
plt.show()

# histogram of out-degrees
plt.hist(list(h_net_seeds_outdegree.values()), bins=9, color='red', alpha=0.7)
plt.xlabel('Out-degree')
plt.ylabel('Frequency')
plt.title('Histogram of out-degrees')
plt.show()

In [ ]:

# bar plot of the degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(h_net_seeds_indegree.keys())), h_net_seeds_indegree.values(), color='red', alpha=0.7)
plt.xticks(range(len(h_net_seeds_indegree.keys())), h_net_seeds_indegree.keys())
plt.xlabel('Node ID')
plt.ylabel('In-degree')
plt.title('In-degree distribution of h_net_seeds')
plt.show()


# bar plot of the degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(h_net_seeds_outdegree.keys())), h_net_seeds_outdegree.values(), color='red', alpha=0.7)
plt.xticks(range(len(h_net_seeds_outdegree.keys())), h_net_seeds_outdegree.keys())
plt.xlabel('Node ID')
plt.ylabel('Out-degree')
plt.title('Out-degree distribution of h_net_seeds')
plt.show()

In [ ]:
# show edges of h_net_sd
edgelist = list(h_net_sd.edges())

# Crea un grafo vuoto
B = nx.Graph()

# Aggiungi gli archi dalla edgelist
B.add_edges_from(edgelist)

# Identifica i due insiemi:
# Assumiamo che i primi elementi delle tuple (nodi seed) appartengano a un insieme
# e i secondi elementi (nodi discovered) appartengano all'altro insieme.
seeds = {edge[0] for edge in edgelist}
discovered = {edge[1] for edge in edgelist}

# Assegna l'attributo 'bipartite' ad ogni nodo:
for nodo in seeds:
    B.nodes[nodo]['bipartite'] = 0  # Insieme 0 (ad es. Stakeholders)
for nodo in discovered:
    B.nodes[nodo]['bipartite'] = 1  # Insieme 1 (Organizzazioni o siti esterni)

In [ ]:
h_net_sd.number_of_nodes(), h_net_sd.number_of_edges()

In [ ]:
# Calcola la disposizione dei nodi (layout bipartito)
pos_sd_h = nx.bipartite_layout(B, seeds)

# Disegna la rete
plt.figure(figsize=(8, 6))
nx.draw(B, pos_sd_h, with_labels=True, node_color=['skyblue' if B.nodes[n].get('bipartite') == 0 else 'lightgreen' for n in B.nodes()], node_size=150, font_size=10)
plt.title("Rete Bipartita: Stakeholders vs Organizzazioni")
plt.show()

In [ ]:
# compute the degree of each seed
seed_degrees = dict(h_net_sd.degree(seeds))
# bar plot of the degree of each seed
plt.figure(figsize=(8, 6))
plt.bar(range(len(seed_degrees.keys())), seed_degrees.values(), color='skyblue')
plt.xticks(range(len(seed_degrees.keys())), seed_degrees.keys())
plt.xlabel('Seed')
plt.ylabel('Degree')
plt.title('Degree of each seed')
plt.show()

# histogram of the degree of seeds
plt.figure(figsize=(8, 6))
plt.hist(seed_degrees.values(), bins=21, color='skyblue')
plt.xlabel('Degree')
plt.ylabel('Frequency')
plt.title('Degree distribution of seeds')
plt.show()

In [ ]:
# compute the degree of each discovered node
discovered_degrees = dict(h_net_sd.degree(discovered))

# histogram of the degree of discovered nodes
plt.figure(figsize=(8, 6))
plt.hist(discovered_degrees.values(), bins=11, color='lightgreen')
plt.yscale('log')
plt.xlabel('Degree')
plt.ylabel('Frequency')
plt.title(f'Degree distribution of discovered nodes, #nodes={len(discovered_degrees)}')
plt.show()

## Projections

In [ ]:
h_seeds = bipartite.projected_graph(B, seeds)
h_seeds_weighted = bipartite.weighted_projected_graph(B, seeds)

h_discovered = bipartite.projected_graph(B, discovered)
h_discovered_weighted = bipartite.weighted_projected_graph(B, discovered)

In [ ]:
# h_discovered is a NetworkX graph
h_discovered.number_of_nodes(), h_discovered.number_of_edges()

In [ ]:
# list of weights of h_discovered
weights_h_discovered = [d['weight'] for u, v, d in h_discovered_weighted.edges(data=True)]
#weights_h_discovered

In [ ]:
# how many weights are equal to 3 in weights_h_discovered
sum(1 for w in weights_h_discovered if w == 3)

In [ ]:
# how many weights are equal to 2 in weights_h_discovered
sum(1 for w in weights_h_discovered if w == 2)

In [ ]:
# how many weights are equal to 1 in weights_h_discovered
sum(1 for w in weights_h_discovered if w == 1)

In [ ]:
# lcc of h_seeds_weighted
lcc_seeds = max(nx.connected_components(h_seeds_weighted), key=len)
h_seeds_weighted = h_seeds_weighted.subgraph(lcc_seeds).copy()

In [ ]:
# number of nodes and edges, average degree and density
def graph_stats_h(G_h):
    n_h = G_h.number_of_nodes()
    m_h = G_h.number_of_edges()
    avg_deg_h = sum(dict(G_h.degree()).values()) / n_h if n_h > 0 else 0
    density_h = nx.density(G_h) if n_h > 1 else 0
    return n_h, m_h, avg_deg_h, density_h

# for G_seeds_bin_h and G_disc_bin_h
n_seeds_h, m_seeds_h, avg_deg_seeds_h, density_seeds_h = graph_stats_h(h_seeds_weighted)
n_disc_h, m_disc_h, avg_deg_disc_h, density_disc_h = graph_stats_h(h_discovered_weighted)

n_seeds_h, m_seeds_h, avg_deg_seeds_h, density_seeds_h

In [ ]:
# Plot the weighted seeds projection graph (h_seeds_weighted)

# import Line2D
from matplotlib.lines import Line2D

# layout specific to this graph (avoid reusing pos_proj which may be for other graphs)
pos_hsw = nx.spring_layout(h_seeds_weighted, seed=42, k=0.6)

# colors/sizes
cat_map = id_to_cat_seeds if 'id_to_cat_seeds' in globals() else seeds_nodes.set_index('ID')['Category'].to_dict()
node_colors = [color_map_cat.get(cat_map.get(n, ''), 'gray') for n in h_seeds_weighted.nodes()]
node_sizes = [450 for _ in h_seeds_weighted.nodes()]

# edge widths scaled by weight
E = list(h_seeds_weighted.edges(data=True))
wts = [d.get('weight', 1) for _, _, d in E]
max_w = max(wts) if wts else 1
edge_widths = [1.2 + 6.0 * (w / max_w) ** 0.7 for w in wts]

plt.figure(figsize=(12, 8))
nx.draw_networkx_nodes(
    h_seeds_weighted, pos_hsw,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors='black',
    linewidths=2.0,
    alpha=0.75
)
nx.draw_networkx_edges(
    h_seeds_weighted, pos_hsw,
    edgelist=E,
    width=edge_widths,
    edge_color='0.35',
    alpha=0.85
)
nx.draw_networkx_labels(
    h_seeds_weighted, pos_hsw,
    labels={n: n for n in h_seeds_weighted.nodes()},
    font_size=12,
    font_weight='bold'
)

# legend
uniq_cats = sorted({cat_map.get(n, '') for n in h_seeds_weighted.nodes() if cat_map.get(n, '')})
legend_elems = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats
]
if legend_elems:
    plt.legend(handles=legend_elems, title='Category',
               loc='upper left', bbox_to_anchor=(1.02, 0.5),
               frameon=False, fontsize=14, title_fontsize=16)

#plt.title('Hyperlink Seed Projection (Weighted)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# plot the graph h_seeds_weighted

In [ ]:
n_disc_h, m_disc_h, avg_deg_disc_h, density_disc_h

In [ ]:
# plot the distribution of weights in h_discovered

plt.style.use('seaborn-v0_8-whitegrid')

max_w = max(weights_h_discovered) if weights_h_discovered else 1
bins = np.arange(0.5, max_w + 1.5, 1) if max_w <= 50 else 50

fig, ax = plt.subplots(figsize=(9, 5), facecolor='#f9fbff')
n, b, patches = ax.hist(
    weights_h_discovered,
    bins=bins,
    color='#4E79A7',          # single color for all bars
    edgecolor='white',
    linewidth=1.2,
    alpha=0.9,
    histtype='bar',
    rwidth=0.5                # make bars narrower
)

ax.set_yscale('log')
#ax.set_title("Distribution of Weights in h_discovered", fontsize=13, pad=12)
ax.set_xlabel("Weight")
ax.set_ylabel("Frequency")
ax.grid(axis='y', linestyle=':', alpha=0.6)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)

# Annotate counts if few bins
if isinstance(bins, np.ndarray) and len(patches) <= 12:
    for patch, count in zip(patches, n):
        if count > 0:
            ax.text(
                patch.get_x() + patch.get_width() / 2,
                max(count * 1.05, 1.01),
                f"{int(count)}",
                ha='center', va='bottom', fontsize=9, color='#2b2d42'
            )

plt.tight_layout()
plt.show()


In [ ]:


# plot the degree distribution of h_discovered_weighted
plt.figure(figsize=(8, 6))
plt.hist(h_discovered_degrees.values(), bins=35, alpha=0.7)
plt.yscale('log')
plt.xlabel('Degree')
plt.ylabel('Frequency')
#plt.title('Degree Distribution of Discovered Nodes (Weighted Projection)')
plt.show()

In [ ]:
# Visualizza la rete proiettata sui nodi seeds (h_seeds) con stile analogo al grafico dei Seed (cell 15)
plt.figure(figsize=(12, 8))

# Posizioni: riusa pos_h se disponibile, altrimenti calcola
#if 'pos_h' in globals():
if False:
    pos_proj = {n: pos_h[n] for n in h_seeds.nodes() if n in pos_h}
    missing = set(h_seeds.nodes()) - set(pos_proj.keys())
    if missing:
        new_pos = nx.spring_layout(h_seeds.subgraph(missing), seed=42)
        pos_proj.update(new_pos)
else:
    pos_proj = nx.spring_layout(h_seeds)#, seed=42)
    #pos_proj = nx.spring_layout(h_net_seeds, k=0.1, iterations=500)

# Colori nodi per categoria
id_to_cat = seeds_nodes.set_index('ID')['Category'].to_dict()
node_colors_proj = [color_map_cat.get(id_to_cat.get(n, ''), 'gray') for n in h_seeds.nodes()]
node_sizes_proj = [450 for _ in h_seeds.nodes()]

nx.draw_networkx_nodes(
    h_seeds, pos_proj,
    node_size=node_sizes_proj,
    node_color=node_colors_proj,
    alpha=0.6,
    edgecolors='black',
    linewidths=2.0
)

nx.draw_networkx_edges(
    h_seeds, pos_proj,
    width=1.8,
    edge_color='0.35',
    alpha=0.8
)

nx.draw_networkx_labels(
    h_seeds, pos_proj,
    labels={n: n for n in h_seeds.nodes()},
    font_size=12,
    font_weight='bold'
)

uniq_cats_proj = sorted({id_to_cat.get(n, '') for n in h_seeds.nodes()})
legend_elements_proj = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats_proj
]
plt.legend(handles=legend_elements_proj, title='Category',
           loc='upper left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Seed Projection Network (h_seeds)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.patches import FancyArrowPatch

# Visualizza la rete proiettata sui nodi seeds (h_seeds_weighted) con margine tra nodi e archi

# mantieni solo la componente connessa principale
h_seeds_weighted = h_seeds_weighted.subgraph(max(nx.connected_components(h_seeds_weighted), key=len))

plt.figure(figsize=(12, 8))

# layout
#pos_proj = nx.spring_layout(h_seeds_weighted, iterations=20)

# colori / size
id_to_cat = seeds_nodes.set_index('ID')['Category'].to_dict()
node_colors_proj = [color_map_cat.get(id_to_cat.get(n, ''), 'gray') for n in h_seeds_weighted.nodes()]
node_sizes_proj = [450 for _ in h_seeds_weighted.nodes()]

nx.draw_networkx_nodes(
    h_seeds_weighted, pos_proj,
    node_size=node_sizes_proj,
    node_color=node_colors_proj,
    alpha=0.6,
    edgecolors='black',
    linewidths=2.0
)

# disegna archi con "shrink" per distanziarli dai nodi

ax = plt.gca()
margin_pts = 15  # aumenta per maggiore distanza (in punti)
for u, v in h_seeds_weighted.edges():
    w = max(1.5, h_seeds_weighted[u][v]['weight'] * 2)
    x1, y1 = pos_proj[u]
    x2, y2 = pos_proj[v]
    patch = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle='-',
        shrinkA=margin_pts,
        shrinkB=margin_pts,
        linewidth=w,
        color='0.35',
        alpha=0.8,
        connectionstyle='arc3'
    )
    patch.set_zorder(1)
    ax.add_patch(patch)

# etichette
nx.draw_networkx_labels(
    h_seeds_weighted, pos_proj,
    labels={n: n for n in h_seeds_weighted.nodes()},
    font_size=12,
    font_weight='bold'
)

# legenda
uniq_cats_proj = sorted({id_to_cat.get(n, '') for n in h_seeds_weighted.nodes()})
legend_elements_proj = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats_proj
]
plt.legend(handles=legend_elements_proj, title='Category',
           loc='upper left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Hyperlink Seed Projection Network')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# distribution of weights in h_discovered
import matplotlib.pyplot as plt

plt.hist(h_discovered.flatten(), bins=50)
plt.title("Distribution of Weights in h_discovered")
plt.xlabel("Weight")
plt.ylabel("Frequency")
plt.show()

In [ ]:
from matplotlib.patches import FancyArrowPatch

# Visualizza la rete proiettata sui nodi seeds (h_discovered) con margine tra nodi e archi

# mantieni solo la componente connessa principale
h_discovered = h_discovered.subgraph(max(nx.connected_components(h_discovered), key=len))

plt.figure(figsize=(12, 8))

# layout
#pos_proj = nx.spring_layout(h_discovered, iterations=20)

# colori / size
id_to_cat = seeds_nodes.set_index('ID')['Category'].to_dict()
node_colors_proj = [color_map_cat.get(id_to_cat.get(n, ''), 'gray') for n in h_discovered.nodes()]
node_sizes_proj = [450 for _ in h_discovered.nodes()]

pos_proj = nx.spring_layout(h_discovered)

nx.draw_networkx_nodes(
    h_discovered, pos_proj,
    node_size=node_sizes_proj,
    node_color=node_colors_proj,
    alpha=0.6,
    edgecolors='black',
    linewidths=2.0
)

# disegna archi con "shrink" per distanziarli dai nodi

ax = plt.gca()
margin_pts = 15  # aumenta per maggiore distanza (in punti)
for u, v in h_discovered.edges():
    w = max(1.5, h_discovered[u][v]['weight'] * 2)
    x1, y1 = pos_proj[u]
    x2, y2 = pos_proj[v]
    patch = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle='-',
        shrinkA=margin_pts,
        shrinkB=margin_pts,
        linewidth=w,
        color='0.35',
        alpha=0.8,
        connectionstyle='arc3'
    )
    patch.set_zorder(1)
    ax.add_patch(patch)

# etichette
nx.draw_networkx_labels(
    h_discovered, pos_proj,
    labels={n: n for n in h_discovered.nodes()},
    font_size=12,
    font_weight='bold'
)

# legenda
uniq_cats_proj = sorted({id_to_cat.get(n, '') for n in h_discovered.nodes()})
legend_elements_proj = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats_proj
]
plt.legend(handles=legend_elements_proj, title='Category',
           loc='upper left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Hyperlink Seed Projection Network')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
h_net_seeds.number_of_nodes(), h_net_seeds.number_of_edges()

In [ ]:
G_filtered.number_of_nodes(), G_filtered.number_of_edges()

In [ ]:
# Soglia desiderata (ad esempio, consideriamo solo archi con weight >= 2)
threshold = 2

# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()

# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(h_net_seeds.nodes(data=True))

# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in h_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=data['weight'])
plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color='skyblue',
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()

In [ ]:
# Soglia desiderata (ad esempio, consideriamo solo archi con weight >= 2)
threshold = 2

# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()

# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(h_net_seeds.nodes(data=True))

# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in h_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=data['weight'])

edges_G1 = set(h_net_seeds.edges())
edges_G2 = set(G_filtered.edges())

intersection = edges_G1.intersection(edges_G2)
union = edges_G1.union(edges_G2)

jaccard_index = len(intersection) / len(union) if union else 0
print("Jaccard Index:", jaccard_index)


# Converte le matrici di adiacenza in array NumPy
A1 = nx.to_numpy_array(h_net_seeds, nodelist=sorted(h_net_seeds.nodes()))
A2 = nx.to_numpy_array(G_filtered, nodelist=sorted(G_filtered.nodes()))

# Calcola la distanza di Hamming
hamming_distance = np.sum(np.abs(A1 - A2))
print("Hamming Distance:", hamming_distance)


plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color='skyblue',
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()

# Twitter

In [ ]:
# Load the data
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')
# Add Status
tw_nodes = tw_nodes.merge(h_nodes[['ID', 'STATUS']], left_on='ID', right_on='ID', how='left')
tw_edges_clean = tw_edges.copy()
# remove where the columns 'Source' and 'Target' are the same
tw_edges_clean = tw_edges_clean[tw_edges_clean['Source'] != tw_edges_clean['Target']]
# remove duplicates
#tw_edges_clean = tw_edges_clean.drop_duplicates(subset=['Source', 'Target'], keep='first')
weighted_tw_edges = tw_edges_clean.copy()
# count the number of edges between each pair of nodes
weighted_tw_edges = weighted_tw_edges.groupby(['Source', 'Target']).size().reset_index(name='weight')

In [ ]:
tw_edges

In [ ]:
tw_edges.groupby('Relationship').size()

In [ ]:
tw_edges_clean.groupby('Relationship').size()

In [ ]:
tw_edges['Date'].hist(bins=460, color='blue', alpha=0.7, figsize=(18, 6))
plt.title(f'Number of Tweets per Week, without self-loops, #tweets={tw_edges.shape[0]}')
plt.xlabel('Date')
plt.ylabel('Number of Tweets')
plt.show()

In [ ]:
tw_edges[tw_edges['Relationship']=='Retweet']['Date'].hist(bins=460, color='blue', alpha=0.7, figsize=(18, 6))
num_retweets = tw_edges[tw_edges['Relationship']=='Retweet'].shape[0]
plt.title(f'Number of Retweets per Week, without self-loops, #retweets={num_retweets}')
plt.xlabel('Date')
plt.ylabel('Number of Tweets')
plt.show()

In [ ]:
tw_net_seeds.number_of_nodes()

In [ ]:
tw_edges.shape

In [ ]:
tw_edges_clean

In [ ]:
# create a directed grapth among only nodes with 'STATUS' == 'Seed'
tw_net_seeds = nx.DiGraph()
tw_net_sd = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    tw_net_sd.add_node(tw_nodes['ID'][i], status=tw_nodes['STATUS'][i], label=tw_nodes['Label'][i], category=tw_nodes['Category'][i])
    if tw_nodes['STATUS'][i] == 'Seed':
        tw_net_seeds.add_node(tw_nodes['ID'][i], status=tw_nodes['STATUS'][i], label=tw_nodes['Label'][i], category=tw_nodes['Category'][i])
# iterate over aggregated edges (weighted_tw_edges has the 'weight' column)
for row in weighted_tw_edges.itertuples(index=False):
    s = row.Source
    t = row.Target
    w = row.weight  # lowercase 'weight' column
    if s in tw_net_seeds.nodes() and t in tw_net_seeds.nodes():
        tw_net_seeds.add_edge(s, t, weight=w)
    else:
        tw_net_sd.add_edge(s, t, weight=w)

# remove self loops
tw_net_seeds.remove_edges_from(nx.selfloop_edges(tw_net_seeds))
#tw_net_sd.remove_edges_from(nx.selfloop_edges(tw_net_sd))




In [ ]:
# plot the distribution of weights in tw_net_sd
weights = [data['weight'] for u, v, data in tw_net_sd.edges(data=True)]
plt.figure(figsize=(8, 6))
plt.hist(weights, bins=30, color='blue', alpha=0.7)
plt.yscale('log')
plt.xlabel('Edge Weight')
plt.ylabel('Frequency')
plt.title(f'Distribution of Edge Weights in tw_net_sd, #edges={len(weights)}')
plt.show()

In [ ]:
# create a directed grapth among only nodes with 'STATUS' == 'Seed'
tw_net_seeds = nx.DiGraph()
tw_net_sd = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    tw_net_sd.add_node(tw_nodes['ID'][i], status=tw_nodes['STATUS'][i], label=tw_nodes['Label'][i], category=tw_nodes['Category'][i])
    if tw_nodes['STATUS'][i] == 'Seed':
        tw_net_seeds.add_node(tw_nodes['ID'][i], status=tw_nodes['STATUS'][i], label=tw_nodes['Label'][i], category=tw_nodes['Category'][i])
# iterate over aggregated edges (weighted_tw_edges has the 'weight' column)
for row in weighted_tw_edges.itertuples(index=False):
    s = row.Source
    t = row.Target
    w = row.weight  # lowercase 'weight' column
    if s in tw_net_seeds.nodes() and t in tw_net_seeds.nodes():
        tw_net_seeds.add_edge(s, t, weight=w)
    else:
        tw_net_sd.add_edge(s, t, weight=w)

# remove self loops
tw_net_seeds.remove_edges_from(nx.selfloop_edges(tw_net_seeds))
#tw_net_sd.remove_edges_from(nx.selfloop_edges(tw_net_sd))

# df: long edgelist with columns ['seed', 'discovered', 'weight']
# For Hyperlink, precompute df['weight']=1 for each edge.

df = pd.DataFrame([(u, v, d['weight']) for u, v, d in tw_net_sd.edges(data=True)], columns=['seed', 'discovered', 'weight'])

# 1) Build biadjacency
B = df.pivot_table(index='seed', columns='discovered',
                   values='weight', aggfunc='sum', fill_value=0).astype(float)

# ----- Option A: Binary projection -----
A = (B > 0).astype(int)
seeds_proj_binary = A.dot(A.T)               # counts of common discovered
disc_proj_binary  = A.T.dot(A)               # counts of common seeds

# ----- Option B: Raw weighted co-occurrence -----
seeds_proj_raw = B.dot(B.T)                  # can be dominated by prolific seeds
disc_proj_raw  = B.T.dot(B)

# ----- Option C (recommended): Row-normalized attention shares -----
row_sum = B.sum(axis=1).replace(0, np.nan)
P = B.div(row_sum, axis=0).fillna(0)         # each seed’s row sums to 1

# Seeds projection: bounded in [0,1]
seeds_proj_share = P.dot(P.T)

# Discovered projection: average co-attention per seed
S_count = P.shape[0]                         # number of seeds
disc_proj_share_avg = P.T.dot(P) / S_count

# (Optional) zero the diagonal if you want only inter-node edges
for M in (seeds_proj_binary, disc_proj_binary,
          seeds_proj_raw, disc_proj_raw,
          seeds_proj_share, disc_proj_share_avg):
    np.fill_diagonal(M.values, 0)




# Build and plot graphs for:
# seeds_proj_binary, seeds_proj_share, seeds_proj_raw,
# disc_proj_binary,  disc_proj_share (alias of disc_proj_share_avg), disc_proj_raw

# 1. Ensure discovered share matrix has expected name
if 'disc_proj_share' not in globals() and 'disc_proj_share_avg' in globals():
    disc_proj_share = disc_proj_share_avg

def matrix_to_graph(M, undirected=True, name=None):
    G = nx.Graph() if undirected else nx.DiGraph()
    nodes = list(M.index)
    for i, u in enumerate(nodes):
        for j in range(i+1, len(nodes)):
            v = nodes[j]
            w = M.iat[i, j]
            if w and w != 0:
                G.add_edge(int(u), int(v), weight=float(w))
    if name:
        G.name = name
    return G

# 2. Build graphs
G_seeds_bin   = matrix_to_graph(seeds_proj_binary, name='seeds_binary')
G_seeds_share = matrix_to_graph(seeds_proj_share, name='seeds_share')
G_seeds_raw   = matrix_to_graph(seeds_proj_raw, name='seeds_raw')

G_disc_bin    = matrix_to_graph(disc_proj_binary, name='disc_binary')
G_disc_share  = matrix_to_graph(disc_proj_share, name='disc_share')
G_disc_raw    = matrix_to_graph(disc_proj_raw, name='disc_raw')

# 3. Category mappings
id_to_cat_seeds = seeds_nodes.set_index('ID')['Category'].to_dict()
if 'tw_nodes' in globals():
    id_to_cat_disc = tw_nodes.set_index('ID')['Category'].to_dict()
elif 'h_nodes' in globals():
    id_to_cat_disc = h_nodes.set_index('ID')['Category'].to_dict()
else:
    id_to_cat_disc = {}

In [ ]:
# --- Assign label and category attributes to projection graphs ---

# Build (or reuse) mapping dicts
if 'id_to_cat_seeds' not in globals():
    id_to_cat_seeds = seeds_nodes.set_index('ID')['Category'].to_dict()
id_to_label_seeds = seeds_nodes.set_index('ID')['Label'].to_dict()

# Discovered (disc) nodes can come from tw_nodes or h_nodes
if 'tw_nodes' in globals():
    id_to_cat_disc_full = tw_nodes.set_index('ID')['Category'].to_dict()
    id_to_label_disc_full = tw_nodes.set_index('ID')['Label'].to_dict()
elif 'h_nodes' in globals():
    id_to_cat_disc_full = h_nodes.set_index('ID')['Category'].to_dict()
    id_to_label_disc_full = h_nodes.set_index('ID')['Label'].to_dict()
else:
    id_to_cat_disc_full = {}
    id_to_label_disc_full = {}

# Fallback to already prepared (possibly subset) dicts if present
id_to_cat_disc = id_to_cat_disc if 'id_to_cat_disc' in globals() else id_to_cat_disc_full
id_to_label_disc = id_to_label_disc_full  # always from full table

def set_attrs(G, label_map, cat_map):
    for n in G.nodes:
        nid = int(n)
        G.nodes[n]['label'] = label_map.get(nid, 'Unknown')
        G.nodes[n]['category'] = cat_map.get(nid, 'Unknown')

# Seeds projections
set_attrs(G_seeds_bin,   id_to_label_seeds, id_to_cat_seeds)
set_attrs(G_seeds_share, id_to_label_seeds, id_to_cat_seeds)
set_attrs(G_seeds_raw,   id_to_label_seeds, id_to_cat_seeds)

# Discovered projections
set_attrs(G_disc_bin,    id_to_label_disc, id_to_cat_disc)
set_attrs(G_disc_share,  id_to_label_disc, id_to_cat_disc)
set_attrs(G_disc_raw,    id_to_label_disc, id_to_cat_disc)


In [ ]:
from collections import Counter

# 6 undirected projection graphs already defined: 
# G_seeds_bin, G_seeds_share, G_seeds_raw, G_disc_bin, G_disc_share, G_disc_raw

proj_graphs = [
    G_seeds_bin, G_seeds_share, G_seeds_raw,
    G_disc_bin,  G_disc_share,  G_disc_raw
]

diversity_broadcast_results = {}

for Gx in proj_graphs:
    if Gx.number_of_nodes() == 0:
        diversity_broadcast_results[Gx.name] = pd.DataFrame(
            columns=['degree','norm_degree','diversity','broadcast','category']
        )
        continue

    # Degree (undirected)
    deg = pd.Series(dict(Gx.degree())).astype(float)
    # Normalized degree (analogous to normalized out-degree in directed case)
    if Gx.number_of_nodes() > 1:
        norm_deg = deg / (Gx.number_of_nodes() - 1)
    else:
        norm_deg = deg * 0.0

    # Neighbor category distributions
    diversity = {}
    for n in Gx.nodes():
        k = deg.get(n, 0.0)
        if k == 0:
            diversity[n] = 0.0
            continue
        neigh_cats = [Gx.nodes[nei].get('category') for nei in Gx.neighbors(n)]
        counts = Counter(neigh_cats)
        total = sum(counts.values())
        if total == 0:
            diversity[n] = 0.0
            continue
        shares = [c / total for c in counts.values()]
        hhi = sum(s*s for s in shares)
        diversity[n] = 1 - hhi  # Simpson diversity (1 - HHI)

    diversity = pd.Series(diversity)

    # Broadcast index = diversity * normalized degree
    broadcast = diversity * norm_deg

    # Store as node attributes
    nx.set_node_attributes(Gx, diversity.to_dict(), 'diversity')
    nx.set_node_attributes(Gx, norm_deg.to_dict(), 'norm_degree')
    nx.set_node_attributes(Gx, broadcast.to_dict(), 'broadcast')

    # Assemble result table
    cat = pd.Series({n: Gx.nodes[n].get('category') for n in Gx.nodes()})
    df_res = pd.DataFrame({
        'degree': deg,
        'norm_degree': norm_deg,
        'diversity': diversity,
        'broadcast': broadcast,
        'category': cat
    }).sort_index()

    diversity_broadcast_results[Gx.name] = df_res

# Example access:
# diversity_broadcast_results['seeds_binary'].head()
# Combined (optional):
projection_diversity_broadcast = pd.concat(
    {k: v for k, v in diversity_broadcast_results.items()},
    names=['graph','node']
)
projection_diversity_broadcast.head()

In [ ]:
G_disc_bin.number_of_nodes(), G_disc_bin.number_of_edges()

In [ ]:
# weights of disc_proj_binary
# edges(data=True) yields (u, v, attr_dict); unpack and read 'weight'
weights_disc_proj_binary = [attr.get('weight', 1) for _, _, attr in G_disc_bin.edges(data=True)]
#weights_disc_proj_binary

In [ ]:
# plot the distribution of weights in disc_proj_binary

plt.style.use('seaborn-v0_8-whitegrid')

max_w = max(weights_disc_proj_binary) if weights_disc_proj_binary else 1
bins = np.arange(0.5, max_w + 1.5, 1) if max_w <= 50 else 50

fig, ax = plt.subplots(figsize=(9, 5), facecolor='#f9fbff')
n, b, patches = ax.hist(
    weights_disc_proj_binary,
    bins=bins,
    color='#4E79A7',
    edgecolor='white',
    linewidth=1.2,
    alpha=0.9,
    histtype='bar',
    rwidth=0.5
)

ax.set_yscale('log')
ax.set_xlabel("Weight")
ax.set_ylabel("Frequency")
ax.grid(axis='y', linestyle=':', alpha=0.6)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)

# Label every integer tick
if isinstance(bins, np.ndarray):
    xticks = np.arange(1, int(max_w) + 1)
    ax.set_xlim(0.5, max_w + 0.5)
    ax.set_xticks(xticks)
    ax.set_xticklabels([str(x) for x in xticks])


plt.tight_layout()
plt.show()


In [ ]:
# overlap the distribution of weights in disc_proj_binary with that of h_discovered
plt.style.use('seaborn-v0_8-whitegrid')

# create a slightly shifted binning for the second histogram (do not overwrite `bins`)
if isinstance(bins, np.ndarray):
    bins2 = bins + 0.18  # slight offset
else:
    max_w_both = max(max(weights_disc_proj_binary or [1]), max(weights_h_discovered or [1]))
    bins = np.arange(0.5, max_w_both + 1.5, 1)
    bins2 = bins + 0.18

fig, ax = plt.subplots(figsize=(9, 5))
fig.patch.set_alpha(0)  # remove background outside the axes

n1, b1, patches1 = ax.hist(
    weights_disc_proj_binary,
    bins=bins,
    color='#4E79A7',
    edgecolor='white',
    linewidth=1.2,
    alpha=0.6,
    histtype='bar',
    rwidth=0.5,
    label='Twitter'
)
n2, b2, patches2 = ax.hist(
    weights_h_discovered,
    bins=bins2,  # use slightly shifted bins
    color='#F28E2B',
    edgecolor='white',
    linewidth=1.2,
    alpha=0.6,
    histtype='bar',
    rwidth=0.5,
    label='Hyperlink'
)
ax.set_yscale('log')
ax.set_xlabel("Weight")
ax.set_ylabel("Frequency")
ax.grid(axis='y', linestyle=':', alpha=0.6)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.legend()

# Label every integer tick
if isinstance(bins, np.ndarray):
    xticks = np.arange(1, int(max_w) + 1)
    ax.set_xlim(0.5, max_w + 0.5)
    ax.set_xticks(xticks)
    ax.set_xticklabels([str(x) for x in xticks])

plt.tight_layout()
plt.show()

In [ ]:
# degree distribution of G_disc_bin
G_disc_bin_degrees = dict(G_disc_bin.degree())
# plot the degree distribution of G_disc_bin
plt.figure(figsize=(8, 6))
plt.hist(G_disc_bin_degrees.values(), bins=35, alpha=0.7)
plt.yscale('log')
plt.xlabel('Degree')
plt.ylabel('Frequency')
#plt.title('Degree Distribution of Discovered Nodes (Twitter Binary Projection)')
plt.show()


In [ ]:
# Degree distributions with wider bins

# reuse precomputed degrees if available; otherwise compute
deg_tw = list(G_disc_bin_degrees.values()) if 'G_disc_bin_degrees' in globals() else [d for _, d in G_disc_bin.degree()]
deg_hl = list(h_discovered_degrees.values()) if 'h_discovered_degrees' in globals() else [d for _, d in h_discovered_weighted.degree()]

plt.style.use('seaborn-v0_8-whitegrid')

# build wider, integer-aligned bins
max_d = max(max(deg_tw or [1]), max(deg_hl or [1]))
target_bins = 50  # aim for ~40 bins
step = max(2, int(np.ceil(max_d / target_bins)))
bins_deg = np.arange(0.5, max_d + step + 0.5, step)
bins_deg2 = bins_deg + step * 0.18  # slight offset

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_alpha(0)

n1, b1, p1 = ax.hist(
    deg_tw,
    bins=bins_deg,
    color='#4E79A7',  # Twitter
    edgecolor='white',
    linewidth=1.2,
    alpha=0.6,
    histtype='bar',
    rwidth=0.8,
    label='Twitter'
)
n2, b2, p2 = ax.hist(
    deg_hl,
    bins=bins_deg2,
    color='#F28E2B',  # Hyperlink
    edgecolor='white',
    linewidth=1.2,
    alpha=0.6,
    histtype='bar',
    rwidth=0.8,
    label='Hyperlink'
)

ax.set_yscale('log')
ax.set_xlabel("Degree")
ax.set_ylabel("Frequency")
ax.grid(axis='y', linestyle=':', alpha=0.6)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.legend()

# tidy x ticks at coarser spacing
tick_step = step * 5
ax.set_xlim(0.5, 351)
ax.set_xticks(np.arange(0, 351, 50))

plt.tight_layout()
plt.show()


In [ ]:
# Double histogram (overlaid) of strengths (weighted degrees):
# Twitter discovered projection (G_disc_bin) vs Hyperlink discovered projection (h_discovered_weighted)

# Compute strengths (sum of edge weights per node)
str_tw = [w for _, w in G_disc_bin.degree(weight='weight')] if 'G_disc_bin' in globals() else []
str_hl = [w for _, w in h_discovered_weighted.degree(weight='weight')] if 'h_discovered_weighted' in globals() else []

plt.style.use('seaborn-v0_8-whitegrid')

if not str_tw and not str_hl:
    print("No strength data to plot.")
else:
    max_s = max(max(str_tw or [1]), max(str_hl or [1]))
    tb = target_bins if 'target_bins' in globals() else 50
    step = max(1, int(np.ceil(max_s / tb)))
    bins_s = np.arange(-0.5, max_s + step + 0.5, step)
    bins_s2 = bins_s + step * 0.18  # slight offset for overlay

    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_alpha(0)

    n1, b1, p1 = ax.hist(
        str_tw,
        bins=bins_s,
        color='#4E79A7',  # Twitter
        edgecolor='white',
        linewidth=1.2,
        alpha=0.6,
        histtype='bar',
        rwidth=0.8,
        label='Twitter'
    )
    n2, b2, p2 = ax.hist(
        str_hl,
        bins=bins_s2,
        color='#F28E2B',  # Hyperlink
        edgecolor='white',
        linewidth=1.2,
        alpha=0.6,
        histtype='bar',
        rwidth=0.8,
        label='Hyperlink'
    )

    ax.set_yscale('log')
    ax.set_xlabel("Strength")
    ax.set_ylabel("Frequency")
    ax.grid(axis='y', linestyle=':', alpha=0.6)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.legend()

    tick_step = step * 5
    xmax = max_s + 0.5
    ax.set_xlim(-0.5, xmax)
    if tick_step > 0:
        ax.set_xticks(np.arange(0, xmax + 1, tick_step))

    plt.tight_layout()
    plt.show()

In [ ]:
# PDF (density-normalized) double histogram of degrees: Twitter (G_disc_bin) vs Hyperlink (h_discovered_weighted)

# Reuse precomputed degree lists if available; otherwise compute
deg_tw = list(G_disc_bin_degrees.values()) if 'G_disc_bin_degrees' in globals() else [d for _, d in G_disc_bin.degree()]
deg_hl = list(h_discovered_degrees.values()) if 'h_discovered_degrees' in globals() else [d for _, d in h_discovered_weighted.degree()]

plt.style.use('seaborn-v0_8-whitegrid')

# Build aligned integer bins and a slightly shifted copy for overlay
tb = target_bins if 'target_bins' in globals() else 50
max_d = max(max(deg_tw or [1]), max(deg_hl or [1]))
step = max(2, int(np.ceil(max_d / tb)))
bins_deg = np.arange(0.5, max_d + step + 0.5, step)
bins_deg2 = bins_deg + step * 0.18  # slight offset for overlay

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_alpha(0)

# density=True gives a probability density (area under the bars ~ 1)
ax.hist(
    deg_tw,
    bins=bins_deg,
    density=True,
    color='#4E79A7',
    edgecolor='white',
    linewidth=1.2,
    alpha=0.6,
    histtype='bar',
    rwidth=0.8,
    label='Twitter'
)
ax.hist(
    deg_hl,
    bins=bins_deg2,
    density=True,
    color='#F28E2B',
    edgecolor='white',
    linewidth=1.2,
    alpha=0.6,
    histtype='bar',
    rwidth=0.8,
    label='Hyperlink'
)

ax.set_xlabel("Degree")
ax.set_ylabel("Probability density")
ax.grid(axis='y', linestyle=':', alpha=0.6)
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
ax.legend()

ax.set_xlim(0.5, max(351, max_d + 0.5))
ax.set_xticks(np.arange(0, max(351, max_d + 0.5), 50))

plt.tight_layout()
plt.show()

In [ ]:
disc_proj_binary

In [ ]:
# distribution of weights in seeds_proj_binary, seeds_proj_share and seeds_proj_raw
plt.figure(figsize=(18, 5))
for i, (M, title) in enumerate(zip(
        [seeds_proj_binary, seeds_proj_share, seeds_proj_raw],
        ['Binary Projection', 'Row-normalized Co-attention', 'Raw Weighted Co-occurrence']
    )):
    plt.subplot(1, 3, i+1)
    weights = M.values[np.triu_indices_from(M.values, k=1)]
    plt.hist(weights, bins=30, color='blue', alpha=0.7)
    plt.yscale('log')
    plt.xlabel('Edge Weight')
    plt.ylabel('Frequency')
    plt.title(f'{title}, #edges={np.sum(weights > 0)}')
plt.tight_layout()
plt.show()

In [ ]:


def edge_widths_from_graph(G, mode):
    weights = [d['weight'] for *_, d in G.edges(data=True)]
    if not weights:
        return []
    w_arr = np.array(weights, dtype=float)
    if mode == 'binary':
        return list(1.2 + 4.0 * (w_arr / w_arr.max())**0.6)
    if mode == 'share':
        return list(1.0 + 6.0 * (w_arr / w_arr.max())**0.7)
    if mode == 'raw':
        return list(1.0 + 3.5 * (np.log1p(w_arr)/np.log1p(w_arr.max()))**0.9)
    return [1.5]*len(weights)

def plot_proj_graphs(graphs, cat_map, title_prefix):
    cols = 3
    plt.figure(figsize=(16, 4.8))
    for idx, (G, mode) in enumerate(graphs, 1):
        if G.number_of_nodes() == 0:
            continue
        # Reuse existing layout if seeds binary (pos_spb) else compute new
        if G.name == 'seeds_binary' and 'pos_spb' in globals():
            pos_local = {n: pos_spb[n] for n in G.nodes() if n in pos_spb}
            missing = set(G.nodes()) - set(pos_local)
            if missing:
                pos_new = nx.spring_layout(G.subgraph(missing), seed=42)
                pos_local.update(pos_new)
        else:
            pos_local = nx.spring_layout(G, seed=42, k=0.6)

        node_colors = [color_map_cat.get(cat_map.get(n, ''), 'gray') for n in G.nodes()]
        ew = edge_widths_from_graph(G, mode)
        plt.subplot(1, cols, idx)
        nx.draw_networkx_nodes(G, pos_local,
                               node_color=node_colors,
                               node_size=450,
                               edgecolors='black',
                               linewidths=1.6,
                               alpha=0.75)
        nx.draw_networkx_edges(G, pos_local,
                               width=ew,
                               edge_color='0.35',
                               alpha=0.85)
        nx.draw_networkx_labels(G, pos_local,
                                labels={n: n for n in G.nodes()},
                                font_size=9,
                                font_weight='bold')
        plt.title(f"{title_prefix} {mode}")
        plt.axis('off')
    # Legend
    uniq_cats_local = sorted({cat_map.get(n, '') for g,_m in graphs for n in g.nodes() if cat_map.get(n,'')})
    legend_elems = [Line2D([0],[0], marker='o', linestyle='', markersize=12,
                           markerfacecolor=color_map_cat.get(c,'gray'),
                           markeredgecolor='black', label=c)
                    for c in uniq_cats_local]
    if legend_elems:
        plt.legend(handles=legend_elems, title='Category',
                   loc='center left', bbox_to_anchor=(1.02, 0.5),
                   frameon=False)
    plt.tight_layout()
    plt.show()

# 4. Plot seeds projections
plot_proj_graphs(
    [
        (G_seeds_bin,   'binary'),
        (G_seeds_share, 'share'),
        (G_seeds_raw,   'raw')
    ],
    id_to_cat_seeds,
    "Seeds projection"
)

# 5. Plot discovered projections
plot_proj_graphs(
    [
        (G_disc_bin,   'binary'),
        (G_disc_share, 'share'),
        (G_disc_raw,   'raw')
    ],
    id_to_cat_disc,
    "Discovered projection"
)

In [ ]:
import math

# Degree (unweighted) distributions for the six projection graphs already built in cell 46

graphs_info = [
    ("Seeds Binary", G_seeds_bin),
    ("Seeds Share",  G_seeds_share),
    ("Seeds Raw",    G_seeds_raw),
    ("Disc Binary",  G_disc_bin),
    ("Disc Share",   G_disc_share),
    ("Disc Raw",     G_disc_raw),
]


def _hist_bins(values):
    if not values:
        return 1
    mx = max(values)
    return min(mx, 20) if mx > 0 else 1

plt.figure(figsize=(14, 7))
for idx, (title, Gx) in enumerate(graphs_info, 1):
    degrees = [d for _n, d in Gx.degree()]
    ax = plt.subplot(2, 3, idx)
    if degrees:
        bins = _hist_bins(degrees)
        ax.hist(degrees, bins=bins, color='steelblue', alpha=0.75, edgecolor='white')
        ax.axvline(np.mean(degrees), color='red', linestyle='--', linewidth=1, label=f"mean={np.mean(degrees):.2f}")
        ax.axvline(np.median(degrees), color='green', linestyle=':', linewidth=1, label=f"med={np.median(degrees):.2f}")
        ax.set_title(f"{title}\nN={Gx.number_of_nodes()} E={Gx.number_of_edges()}")
        ax.set_xlabel("Degree")
        ax.set_ylabel("Frequency")
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "Empty graph", ha='center', va='center')
        ax.set_title(title)
plt.tight_layout()
plt.show()

# (Optional) Strength (weighted degree) distributions (useful for non-binary variants)
plt.figure(figsize=(14, 7))
for idx, (title, Gx) in enumerate(graphs_info, 1):
    strengths = [w for _n, w in Gx.degree(weight='weight')]
    ax = plt.subplot(2, 3, idx)
    if strengths and any(s > 0 for s in strengths):
        # Log-scale bins for heavy-tailed weights
        positive = [s for s in strengths if s > 0]
        nbins = 18
        mn, mx = min(positive), max(positive)
        if mx > mn:
            bins = np.logspace(math.log10(mn), math.log10(mx), nbins)
        else:
            bins = 10
        ax.hist(positive, bins=bins, color='slateblue', alpha=0.75, edgecolor='white')
        ax.set_xscale('log')
        ax.axvline(np.mean(positive), color='red', linestyle='--', linewidth=1, label=f"mean={np.mean(positive):.1f}")
        ax.axvline(np.median(positive), color='green', linestyle=':', linewidth=1, label=f"med={np.median(positive):.1f}")
        ax.set_title(f"{title} (strength)")
        ax.set_xlabel("Strength (weighted degree, log scale)")
        ax.set_ylabel("Frequency")
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "No positive weights", ha='center', va='center')
        ax.set_title(f"{title} (strength)")
plt.tight_layout()
plt.show()

In [ ]:
# Metrics for projection graphs (cell 46 already created the graphs)
# Uses existing: G_seeds_bin, G_seeds_share, G_seeds_raw, G_disc_bin, G_disc_share, G_disc_raw
# and category maps: id_to_cat_seeds, id_to_cat_disc

def projection_metrics(G, cat_map=None):
    if G.number_of_nodes() == 0:
        return pd.DataFrame(columns=[
            'degree','strength','clustering_unweighted','triangles','betweenness','category'
        ])
    deg = dict(G.degree())
    strength = dict(G.degree(weight='weight'))
    clustering = nx.clustering(G, weight=None)          # unweighted clustering
    triangles = nx.triangles(G)                         # number of triangles each node participates in
    betw = nx.betweenness_centrality(G, weight='weight', normalized=True)
    rows = []
    for n in G.nodes():
        rows.append({
            'node': int(n),
            'degree': deg.get(n, 0),
            'strength': strength.get(n, 0),
            'clustering_unweighted': clustering.get(n, 0.0),
            'triangles': triangles.get(n, 0),
            'betweenness': betw.get(n, 0.0),
            'category': cat_map.get(n) if cat_map else None
        })
    return pd.DataFrame(rows).set_index('node').sort_index()

metrics_seeds_binary = projection_metrics(G_seeds_bin,   id_to_cat_seeds)
metrics_seeds_share  = projection_metrics(G_seeds_share, id_to_cat_seeds)
metrics_seeds_raw    = projection_metrics(G_seeds_raw,   id_to_cat_seeds)

metrics_disc_binary  = projection_metrics(G_disc_bin,   id_to_cat_disc)
metrics_disc_share   = projection_metrics(G_disc_share, id_to_cat_disc)
metrics_disc_raw     = projection_metrics(G_disc_raw,   id_to_cat_disc)

projection_metrics_dfs = {
    'seeds_proj_binary': metrics_seeds_binary,
    'seeds_proj_share':  metrics_seeds_share,
    'seeds_proj_raw':    metrics_seeds_raw,
    'disc_proj_binary':  metrics_disc_binary,
    'disc_proj_share':   metrics_disc_share,
    'disc_proj_raw':     metrics_disc_raw
}

# Example display (remove or change as needed)
metrics_seeds_binary

In [ ]:
from collections import Counter

# Extended metrics including label (first col), category (second), diversity, broadcast

def projection_metrics_extended(G, label_map, cat_map):
    if G.number_of_nodes() == 0:
        cols = ['label','category','degree','strength','diversity','broadcast',
                'clustering_unweighted','triangles','betweenness']
        return pd.DataFrame(columns=cols)
    deg = dict(G.degree())
    strength = dict(G.degree(weight='weight'))
    clustering = nx.clustering(G, weight=None)
    triangles = nx.triangles(G)
    betw = nx.betweenness_centrality(G, weight='weight', normalized=True)

    rows = []
    for n in G.nodes():
        # diversity & broadcast already stored as node attributes in cell 46
        div = G.nodes[n].get('diversity')
        brd = G.nodes[n].get('broadcast')
        # fallback (only if missing)
        if div is None or brd is None:
            neigh_cats = [G.nodes[nb].get('category') for nb in G.neighbors(n)]
            k = len(neigh_cats)
            if k:
                counts = Counter(neigh_cats)
                shares = [c/k for c in counts.values()]
                hhi = sum(s*s for s in shares)
                div = 1 - hhi
            else:
                div = 0.0
            brd = div * (deg.get(n,0)/(G.number_of_nodes()-1)) if G.number_of_nodes()>1 else 0.0

        rows.append({
            'node': int(n),
            'label': label_map.get(int(n), 'Unknown'),
            'category': cat_map.get(int(n), 'Unknown'),
            'degree': deg.get(n, 0),
            'strength': strength.get(n, 0),
            'diversity': div,
            'broadcast': brd,
            'clustering_unweighted': clustering.get(n, 0.0),
            'triangles': triangles.get(n, 0),
            'betweenness': betw.get(n, 0.0)
        })
    df = pd.DataFrame(rows).set_index('node').sort_index()
    # Reorder columns explicitly
    return df[['label','category','degree','strength','diversity','broadcast',
               'clustering_unweighted','triangles','betweenness']]

# Build updated metrics dict
projection_metrics_dfs = {
    'seeds_proj_binary': projection_metrics_extended(G_seeds_bin,   id_to_label_seeds, id_to_cat_seeds),
    'seeds_proj_share':  projection_metrics_extended(G_seeds_share, id_to_label_seeds, id_to_cat_seeds),
    'seeds_proj_raw':    projection_metrics_extended(G_seeds_raw,   id_to_label_seeds, id_to_cat_seeds),
    'disc_proj_binary':  projection_metrics_extended(G_disc_bin,    id_to_label_disc,  id_to_cat_disc),
    'disc_proj_share':   projection_metrics_extended(G_disc_share,  id_to_label_disc,  id_to_cat_disc),
    'disc_proj_raw':     projection_metrics_extended(G_disc_raw,    id_to_label_disc,  id_to_cat_disc),
}

# Example display
projection_metrics_dfs['seeds_proj_binary']

In [ ]:
from collections import Counter

# Extended metrics including label (first col), category (second), diversity, broadcast

def projection_metrics_extended(G, label_map, cat_map):
    if G.number_of_nodes() == 0:
        cols = ['label','category','degree','strength','diversity','broadcast', 'clustering_weig.',
                'clustering_unweig.','triangles','betweenness']
        return pd.DataFrame(columns=cols)
    deg = dict(G.degree())
    strength = dict(G.degree(weight='weight'))
    clustering_u = nx.clustering(G, weight=None)
    clustering_v = nx.clustering(G, weight='weight')
    triangles = nx.triangles(G)
    betw = nx.betweenness_centrality(G, weight='weight', normalized=True)

    rows = []
    for n in G.nodes():
        # diversity & broadcast already stored as node attributes in cell 46
        div = G.nodes[n].get('diversity')
        brd = G.nodes[n].get('broadcast')
        # fallback (only if missing)
        if div is None or brd is None:
            neigh_cats = [G.nodes[nb].get('category') for nb in G.neighbors(n)]
            k = len(neigh_cats)
            if k:
                counts = Counter(neigh_cats)
                shares = [c/k for c in counts.values()]
                hhi = sum(s*s for s in shares)
                div = 1 - hhi
            else:
                div = 0.0
            brd = div * (deg.get(n,0)/(G.number_of_nodes()-1)) if G.number_of_nodes()>1 else 0.0

        rows.append({
            'node': int(n),
            'label': label_map.get(int(n), 'Unknown'),
            'category': cat_map.get(int(n), 'Unknown'),
            'degree': deg.get(n, 0),
            'strength': strength.get(n, 0),
            'diversity': div,
            'broadcast': brd,
            'clustering_weig.': clustering_v.get(n, 0.0),
            'clustering_unweig.': clustering_u.get(n, 0.0),
            'triangles': triangles.get(n, 0),
            'betweenness': betw.get(n, 0.0)
        })
    df = pd.DataFrame(rows).set_index('node').sort_index()
    # Reorder columns explicitly
    return df[['label','category','degree','strength','diversity','broadcast','clustering_weig.',
               'clustering_unweig.','triangles','betweenness']]

# Build updated metrics dict
projection_metrics_dfs = {
    'seeds_proj_binary': projection_metrics_extended(G_seeds_bin,   id_to_label_seeds, id_to_cat_seeds),
    'seeds_proj_share':  projection_metrics_extended(G_seeds_share, id_to_label_seeds, id_to_cat_seeds),
    'seeds_proj_raw':    projection_metrics_extended(G_seeds_raw,   id_to_label_seeds, id_to_cat_seeds),
    'disc_proj_binary':  projection_metrics_extended(G_disc_bin,    id_to_label_disc,  id_to_cat_disc),
    'disc_proj_share':   projection_metrics_extended(G_disc_share,  id_to_label_disc,  id_to_cat_disc),
    'disc_proj_raw':     projection_metrics_extended(G_disc_raw,    id_to_label_disc,  id_to_cat_disc),
}

# Example display
projection_metrics_dfs['seeds_proj_binary']

In [ ]:
projection_metrics_dfs['seeds_proj_share']

In [ ]:
projection_metrics_dfs['seeds_proj_raw']

In [ ]:
projection_metrics_dfs['disc_proj_binary']

In [ ]:
projection_metrics_dfs['disc_proj_share']

In [ ]:
projection_metrics_dfs['disc_proj_raw']

In [ ]:
from matplotlib.patches import FancyArrowPatch

# Build undirected graph from the binary co-occurrence dataframe
G_spb = nx.Graph()
for u in seeds_proj_binary.index:
    for v in seeds_proj_binary.columns:
        if u < v:  # avoid duplicates / diagonal
            w = seeds_proj_binary.loc[u, v]
            if w > 0:
                G_spb.add_edge(int(u), int(v), weight=int(w))
plt.figure(figsize=(12, 8))

# layout
# Layout (reuse if already computed)
#if 'pos_spb' in globals():
#    pos_proj = pos_spb
#else:
pos_proj = nx.spring_layout(G_spb, seed=42)

# Categories and colors
id_to_cat = seeds_nodes.set_index('ID')['Category'].to_dict()
node_colors_proj = [color_map_cat.get(id_to_cat.get(n, ''), 'gray') for n in G_spb.nodes()]
node_sizes_proj = [450 for _ in G_spb.nodes()]

# Draw nodes
nx.draw_networkx_nodes(
    G_spb, pos_proj,
    node_size=node_sizes_proj,
    node_color=node_colors_proj,
    alpha=0.6,
    edgecolors='black',
    linewidths=2.0
)

# Draw weighted edges with margin (undirected)
ax = plt.gca()
margin_pts = 12
max_w = max([d['weight'] for *_, d in G_spb.edges(data=True)], default=1)
for u, v, d in G_spb.edges(data=True):
    w = d['weight']
    lw = 1.2 + 7.0 * (w / max_w)
    x1, y1 = pos_proj[u]
    x2, y2 = pos_proj[v]
    patch = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle='-',
        shrinkA=margin_pts,
        shrinkB=margin_pts,
        linewidth=lw,
        color='0.35',
        alpha=0.85,
        connectionstyle='arc3'
    )
    patch.set_zorder(1)
    ax.add_patch(patch)

# Labels
nx.draw_networkx_labels(
    G_spb, pos_proj,
    labels={n: n for n in G_spb.nodes()},
    font_size=12,
    font_weight='bold'
)

# Legend
uniq_cats_proj = sorted({id_to_cat.get(n, '') for n in G_spb.nodes()})
legend_elements_proj = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats_proj
]
plt.legend(handles=legend_elements_proj, title='Category',
           loc='upper left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Seed Binary Co-occurrence Network ')
#plt.title('Seed Binary Projection Network (Common Discovered Count)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Plot seeds_proj_binary (undirected co-occurrence of seeds) with style similar to prior seed graphs
from matplotlib.lines import Line2D  # Needed for custom legend (fix NameError)

# Build undirected graph from the binary co-occurrence dataframe
G_spb = nx.Graph()
for u in seeds_proj_binary.index:
    for v in seeds_proj_binary.columns:
        if u < v:  # avoid duplicates / diagonal
            w = seeds_proj_binary.loc[u, v]
            if w > 0:
                G_spb.add_edge(int(u), int(v), weight=int(w))

# Node categories (reuse existing mapping)
id_to_cat = seeds_nodes.set_index('ID')['Category'].to_dict()

# Reuse prior seed projection layout if available; else compute
if 'pos_proj' in globals():
    pos_spb = {n: pos_proj[n] for n in G_spb.nodes() if n in pos_proj}
    missing = set(G_spb.nodes()) - set(pos_spb.keys())
    if missing:
        pos_missing = nx.spring_layout(G_spb.subgraph(missing), seed=42)
        pos_spb.update(pos_missing)
else:
    pos_spb = nx.spring_layout(G_spb, seed=42, k=0.6)

# Node colors / sizes
node_colors = [color_map_cat.get(id_to_cat.get(n, ''), 'gray') for n in G_spb.nodes()]
node_sizes = [450 for _ in G_spb.nodes()]

# Edge widths scaled by weight
if G_spb.number_of_edges() > 0:
    weights = np.array([d['weight'] for *_, d in G_spb.edges(data=True)])
    max_w = weights.max()
    edge_widths = [1.2 + 3.0 * (d['weight'] / max_w) ** 0.6 for *_, d in G_spb.edges(data=True)]
else:
    edge_widths = []

plt.figure(figsize=(12, 8))

nx.draw_networkx_nodes(
    G_spb, pos_spb,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors='black',
    linewidths=2.0,
    alpha=0.7
)

nx.draw_networkx_edges(
    G_spb, pos_spb,
    width=edge_widths,
    edge_color='0.35',
    alpha=0.85
)

nx.draw_networkx_labels(
    G_spb, pos_spb,
    labels={n: n for n in G_spb.nodes()},
    font_size=12,
    font_weight='bold'
)

# Legend
uniq_cats = sorted({id_to_cat.get(n, '') for n in G_spb.nodes()})
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats
]
plt.legend(handles=legend_elements, title='Category',
           loc='upper left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Seed Binary Projection Network (Common Discovered Count)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# print top 10 nodes by degree in tw_net_seeds
top_10_seeds = sorted(tw_net_seeds.degree, key=lambda x: x[1], reverse=True)[:10]
print("Top 10 nodes by degree in tw_net_seeds:")
for node, degree in top_10_seeds:
    print(f"Node: {node}, Degree: {degree}")

In [ ]:
# print top 10 nodes by strength in tw_net_seeds
top_10_seeds = sorted(tw_net_seeds.degree(weight='weight'), key=lambda x: x[1], reverse=True)[:10]
print("Top 10 nodes by strength in tw_net_seeds:")
for node, strength in top_10_seeds:
    print(f"Node: {node}, Strength: {strength}")

In [ ]:
# create a directed grapth among only nodes with 'STATUS' == 'Seed'
tw_net_seeds = nx.DiGraph()
tw_net_sd = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    tw_net_sd.add_node(tw_nodes['ID'][i], status=tw_nodes['STATUS'][i], label=tw_nodes['Label'][i], category=tw_nodes['Category'][i])
    if tw_nodes['STATUS'][i] == 'Seed':
        tw_net_seeds.add_node(tw_nodes['ID'][i], status=tw_nodes['STATUS'][i], label=tw_nodes['Label'][i], category=tw_nodes['Category'][i])
for i in range(tw_edges.shape[0]):
    if tw_edges['Source'][i] in tw_net_seeds.nodes() and tw_edges['Target'][i] in tw_net_seeds.nodes():
        tw_net_seeds.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])
    else:
        tw_net_sd.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])

# remove self loops
tw_net_seeds.remove_edges_from(nx.selfloop_edges(tw_net_seeds))
#tw_net_sd.remove_edges_from(nx.selfloop_edges(tw_net_sd))




In [ ]:
# print top 10 nodes by degree in tw_net_seeds
top_10_seeds = sorted(tw_net_seeds.degree, key=lambda x: x[1], reverse=True)[:10]
print("Top 10 nodes by degree in tw_net_seeds:")
for node, degree in top_10_seeds:
    print(f"Node: {node}, Degree: {degree}")

In [ ]:
tw_edges_clean.groupby(['Source', 'Target']).size().reset_index(name='counts').sort_values('counts', ascending=False)['counts'].hist(bins=50)
plt.yscale('log')

In [ ]:
tw_edges.groupby(['Source', 'Target']).size().reset_index(name='counts').sort_values('counts', ascending=False)['counts'].mean()

In [ ]:
# avg degree
avg_degree = sum(dict(tw_net_seeds.degree()).values()) / tw_net_seeds.number_of_nodes()
print(f"Average degree in tw_net_seeds: {avg_degree:.2f}")

In [ ]:
# create a directed graptw among only nodes with 'STATUS' == 'IN' only if Relationship == 'Retweet'
rtw_net_seeds = nx.DiGraph()
rtw_net_sd = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    rtw_net_sd.add_node(tw_nodes['ID'][i])
    if tw_nodes['STATUS'][i] == 'IN':
        rtw_net_seeds.add_node(tw_nodes['ID'][i])
for i in range(tw_edges.shape[0]):
    if tw_edges['Source'][i] in rtw_net_seeds.nodes() and tw_edges['Target'][i] in rtw_net_seeds.nodes() and tw_edges['Relationship'][i] == 'Retweet':
        rtw_net_seeds.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])
    elif tw_edges['Relationship'][i] == 'Retweet':
        rtw_net_sd.add_edge(tw_edges['Source'][i], tw_edges['Target'][i])

# remove self loops
rtw_net_seeds.remove_edges_from(nx.selfloop_edges(rtw_net_seeds))
rtw_net_sd.remove_edges_from(nx.selfloop_edges(rtw_net_sd))




In [ ]:
# 

In [ ]:
tw_net_seeds.number_of_nodes(), tw_net_seeds.number_of_edges()

In [ ]:
# Start from a spring layout
pos_tw = nx.spring_layout(tw_net_seeds, k=0.5, iterations=60)

# Copy so we can modify
pos_tw_adj = {n: pos_tw[n].copy() for n in pos_tw}

min_dist = 0.1   # desired minimum distance between close nodes (tune)
outer_shrink = 0.97  # global inward factor per iteration for distant nodes
iters = 25

nodes = list(pos_tw_adj.keys())

for _ in range(iters):
    moved = False
    # Push very close pairs apart
    for i in range(len(nodes)):
        for j in range(i+1, len(nodes)):
            u, v = nodes[i], nodes[j]
            vec = pos_tw_adj[u] - pos_tw_adj[v]
            dist = np.linalg.norm(vec)
            if dist < 1e-9:
                continue
            if dist < min_dist:
                # amount to push so they reach min_dist
                shift = 0.5 * (min_dist - dist) * vec / dist
                pos_tw_adj[u] += shift
                pos_tw_adj[v] -= shift
                moved = True
    # Mild overall contraction to pull far nodes a bit closer
    if moved:
        # center before shrinking
        coord_arr = np.array(list(pos_tw_adj.values()))
        center = coord_arr.mean(axis=0)
        for n in nodes:
            pos_tw_adj[n] = center + (pos_tw_adj[n] - center) * outer_shrink
    else:
        break  # no close pairs left

# Optional radial non-linear rescale: expand inner radii, compress outer radii
coord_arr = np.array(list(pos_tw_adj.values()))
center = coord_arr.mean(axis=0)
rel = coord_arr - center
r = np.linalg.norm(rel, axis=1)
max_r = r.max() if r.max() > 0 else 1.0
r_norm = r / max_r
# Piecewise: expand small (<0.5), compress large (>0.5)
new_r_norm = np.where(r_norm < 0.5, r_norm**0.7, 0.5 + (r_norm - 0.5) * 0.6)
scale_factors = np.divide(new_r_norm, r_norm, out=np.ones_like(r_norm), where=r_norm>0)
rel = rel * scale_factors[:, None]
for idx, n in enumerate(nodes):
    pos_tw_adj[n] = center + rel[idx]

# Use adjusted positions for plotting
pos_tw = pos_tw_adj

from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

plt.figure(figsize=(12, 8))

node_colors = [color_map_cat.get(tw_net_seeds.nodes[n]['category'], 'gray') for n in tw_net_seeds.nodes()]
node_sizes = [450 for _ in tw_net_seeds.nodes()]

nx.draw_networkx_nodes(
    tw_net_seeds, pos_tw,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.6,
    edgecolors='black',
    linewidths=2.0
)

scale = 1.7
min_width = 1.2
#edge_widths = [
    #max(min_width, scale * np.log1p(tw_net_seeds[u][v].get('weight', 1)))
    #for u, v in tw_net_seeds.edges()]

# unweighted edges
edge_widths = [max(min_width, scale * 1.0) for _ in tw_net_seeds.edges()]


arrows = nx.draw_networkx_edges(
    tw_net_seeds, pos_tw,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,
    connectionstyle='arc3,rad=0.12',
    min_source_margin=2,
    min_target_margin=2
)

nx.draw_networkx_labels(
    tw_net_seeds, pos_tw,
    labels={n: n for n in tw_net_seeds.nodes()},
    font_size=12,
    font_weight='bold'
)

try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

uniq_cats = sorted({tw_net_seeds.nodes[n]['category'] for n in tw_net_seeds.nodes()})
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats
]
plt.legend(handles=legend_elements, title='Category',
           loc='upper left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Seed Twitter Network')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Start from a spring layout
#pos_tw = nx.spring_layout(tw_net_seeds, k=0.5, iterations=60)

# Copy so we can modify
#pos_tw_adj = {n: pos_tw[n].copy() for n in pos_tw}

min_dist = 0.1   # desired minimum distance between close nodes (tune)
outer_shrink = 0.97  # global inward factor per iteration for distant nodes
iters = 25

nodes = list(pos_tw_adj.keys())

for _ in range(iters):
    moved = False
    # Push very close pairs apart
    for i in range(len(nodes)):
        for j in range(i+1, len(nodes)):
            u, v = nodes[i], nodes[j]
            vec = pos_tw_adj[u] - pos_tw_adj[v]
            dist = np.linalg.norm(vec)
            if dist < 1e-9:
                continue
            if dist < min_dist:
                # amount to push so they reach min_dist
                shift = 0.5 * (min_dist - dist) * vec / dist
                pos_tw_adj[u] += shift
                pos_tw_adj[v] -= shift
                moved = True
    # Mild overall contraction to pull far nodes a bit closer
    if moved:
        # center before shrinking
        coord_arr = np.array(list(pos_tw_adj.values()))
        center = coord_arr.mean(axis=0)
        for n in nodes:
            pos_tw_adj[n] = center + (pos_tw_adj[n] - center) * outer_shrink
    else:
        break  # no close pairs left

# Optional radial non-linear rescale: expand inner radii, compress outer radii
coord_arr = np.array(list(pos_tw_adj.values()))
center = coord_arr.mean(axis=0)
rel = coord_arr - center
r = np.linalg.norm(rel, axis=1)
max_r = r.max() if r.max() > 0 else 1.0
r_norm = r / max_r
# Piecewise: expand small (<0.5), compress large (>0.5)
new_r_norm = np.where(r_norm < 0.5, r_norm**0.7, 0.5 + (r_norm - 0.5) * 0.6)
scale_factors = np.divide(new_r_norm, r_norm, out=np.ones_like(r_norm), where=r_norm>0)
rel = rel * scale_factors[:, None]
for idx, n in enumerate(nodes):
    pos_tw_adj[n] = center + rel[idx]

# Use adjusted positions for plotting
pos_tw = pos_tw_adj

from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

plt.figure(figsize=(12, 8))

node_colors = [color_map_cat.get(tw_net_seeds.nodes[n]['category'], 'gray') for n in tw_net_seeds.nodes()]
node_sizes = [450 for _ in tw_net_seeds.nodes()]

nx.draw_networkx_nodes(
    tw_net_seeds, pos_tw,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.6,
    edgecolors='black',
    linewidths=2.0
)

scale = 1.7
min_width = 1.2
#edge_widths = [
    #max(min_width, scale * np.log1p(tw_net_seeds[u][v].get('weight', 1)))
    #for u, v in tw_net_seeds.edges()]

# unweighted edges
edge_widths = [max(min_width, scale * 1.0) for _ in tw_net_seeds.edges()]


arrows = nx.draw_networkx_edges(
    tw_net_seeds, pos_tw,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,
    connectionstyle='arc3,rad=0.12',
    min_source_margin=2,
    min_target_margin=2
)

nx.draw_networkx_labels(
    tw_net_seeds, pos_tw,
    labels={n: n for n in tw_net_seeds.nodes()},
    font_size=12,
    font_weight='bold'
)

try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

uniq_cats = sorted({tw_net_seeds.nodes[n]['category'] for n in tw_net_seeds.nodes()})
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats
]

plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Start from a spring layout
pos_tw = nx.spring_layout(tw_net_seeds, k=0.5, iterations=60)

# Copy so we can modify
pos_tw_adj = {n: pos_tw[n].copy() for n in pos_tw}

min_dist = 0.1   # desired minimum distance between close nodes (tune)
outer_shrink = 0.97  # global inward factor per iteration for distant nodes
iters = 25

nodes = list(pos_tw_adj.keys())

for _ in range(iters):
    moved = False
    # Push very close pairs apart
    for i in range(len(nodes)):
        for j in range(i+1, len(nodes)):
            u, v = nodes[i], nodes[j]
            vec = pos_tw_adj[u] - pos_tw_adj[v]
            dist = np.linalg.norm(vec)
            if dist < 1e-9:
                continue
            if dist < min_dist:
                # amount to push so they reach min_dist
                shift = 0.5 * (min_dist - dist) * vec / dist
                pos_tw_adj[u] += shift
                pos_tw_adj[v] -= shift
                moved = True
    # Mild overall contraction to pull far nodes a bit closer
    if moved:
        # center before shrinking
        coord_arr = np.array(list(pos_tw_adj.values()))
        center = coord_arr.mean(axis=0)
        for n in nodes:
            pos_tw_adj[n] = center + (pos_tw_adj[n] - center) * outer_shrink
    else:
        break  # no close pairs left

# Optional radial non-linear rescale: expand inner radii, compress outer radii
coord_arr = np.array(list(pos_tw_adj.values()))
center = coord_arr.mean(axis=0)
rel = coord_arr - center
r = np.linalg.norm(rel, axis=1)
max_r = r.max() if r.max() > 0 else 1.0
r_norm = r / max_r
# Piecewise: expand small (<0.5), compress large (>0.5)
new_r_norm = np.where(r_norm < 0.5, r_norm**0.7, 0.5 + (r_norm - 0.5) * 0.6)
scale_factors = np.divide(new_r_norm, r_norm, out=np.ones_like(r_norm), where=r_norm>0)
rel = rel * scale_factors[:, None]
for idx, n in enumerate(nodes):
    pos_tw_adj[n] = center + rel[idx]

# Use adjusted positions for plotting
pos_tw = pos_tw_adj

from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

plt.figure(figsize=(12, 8))

node_colors = [color_map_cat.get(tw_net_seeds.nodes[n]['category'], 'gray') for n in tw_net_seeds.nodes()]
node_sizes = [450 for _ in tw_net_seeds.nodes()]

nx.draw_networkx_nodes(
    tw_net_seeds, pos_tw,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.6,
    edgecolors='black',
    linewidths=2.0
)

scale = 1.7
min_width = 1.2
edge_widths = [
    max(min_width, scale * np.log1p(tw_net_seeds[u][v].get('weight', 1)))
    for u, v in tw_net_seeds.edges()
]

arrows = nx.draw_networkx_edges(
    tw_net_seeds, pos_tw,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,
    connectionstyle='arc3,rad=0.12',
    min_source_margin=2,
    min_target_margin=2
)

nx.draw_networkx_labels(
    tw_net_seeds, pos_tw,
    labels={n: n for n in tw_net_seeds.nodes()},
    font_size=12,
    font_weight='bold'
)

try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

uniq_cats = sorted({tw_net_seeds.nodes[n]['category'] for n in tw_net_seeds.nodes()})
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats
]
plt.legend(handles=legend_elements, title='Category',
           loc='upper left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Seed Twitter Network (Adjusted distances)')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:


from matplotlib import patheffects as pe
from matplotlib.lines import Line2D

plt.figure(figsize=(12, 8))

node_colors = [color_map_cat.get(tw_net_seeds.nodes[n]['category'], 'gray') for n in tw_net_seeds.nodes()]
node_sizes = [450 for _ in tw_net_seeds.nodes()]

nx.draw_networkx_nodes(
    tw_net_seeds, pos_tw,
    node_size=node_sizes,
    node_color=node_colors,
    alpha=0.6,
    edgecolors='black',
    linewidths=2.0
)

scale = 1.7
min_width = 1.2
edge_widths = [
    max(min_width, scale * np.log1p(tw_net_seeds[u][v].get('weight', 1)))
    for u, v in tw_net_seeds.edges()
]

arrows = nx.draw_networkx_edges(
    tw_net_seeds, pos_tw,
    width=edge_widths,
    alpha=0.8,
    edge_color='0.35',
    arrowstyle='-|>',
    arrowsize=28,
    connectionstyle='arc3,rad=0.12',
    min_source_margin=2,
    min_target_margin=2
)

nx.draw_networkx_labels(
    tw_net_seeds, pos_tw,
    labels={n: n for n in tw_net_seeds.nodes()},
    font_size=12,
    font_weight='bold'
)

try:
    for a in arrows:
        a.set_path_effects([pe.Stroke(linewidth=a.get_linewidth() + 1.2, foreground='white'), pe.Normal()])
except TypeError:
    arrows.set_path_effects([pe.Stroke(linewidth=2.5, foreground='white'), pe.Normal()])

uniq_cats = sorted({tw_net_seeds.nodes[n]['category'] for n in tw_net_seeds.nodes()})
legend_elements = [
    Line2D([0], [0], marker='o', linestyle='', markersize=16,
           markerfacecolor=color_map_cat.get(c, 'gray'),
           markeredgecolor='black', label=c)
    for c in uniq_cats
]
plt.legend(handles=legend_elements, title='Category',
           loc='upper left', bbox_to_anchor=(1.02, 0.5),
           frameon=False, fontsize=14, title_fontsize=16)

plt.title('Seed Twitter Network')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# plot h_net_seeds with color based on stance
node_colors = []
for i, node in enumerate(tw_net_seeds.nodes()):
    if seeds_nodes.iloc[i]['Stance'] == 1:
        node_colors.append('green')
    elif seeds_nodes.iloc[i]['Stance'] == 0:
        node_colors.append('red')
plt.figure(figsize=(5, 5))
plt.title('Twitter Network of Seeds with Stance towards the Off-Shore Wind Farm')
pos = nx.spring_layout(tw_net_seeds)
nx.draw(tw_net_seeds, pos, node_size=350, node_color=node_colors, edge_color='gray', with_labels=True)
plt.show()


In [ ]:
# compute the degree of each node in tw_net_seeds
tw_net_seeds_indegree = dict(tw_net_seeds.in_degree())

# compute the out-degree of each node in tw_net_seeds
tw_net_seeds_outdegree = dict(tw_net_seeds.out_degree())

# histogram of in-degrees
plt.hist(list(tw_net_seeds_indegree.values()), bins=19, color='red', alpha=0.7)
plt.xlabel('In-degree')
plt.ylabel('Frequency')
plt.title('Histogram of in-degrees')
plt.show()

# histogram of out-degrees
plt.hist(list(tw_net_seeds_outdegree.values()), bins=19, color='red', alpha=0.7)
plt.xlabel('Out-degree')
plt.ylabel('Frequency')
plt.title('Histogram of out-degrees')
plt.show()

In [ ]:
# bar plot of the in-degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(tw_net_seeds_indegree.keys())), tw_net_seeds_indegree.values(), color='red', alpha=0.7)
plt.xticks(range(len(tw_net_seeds_indegree.keys())), tw_net_seeds_indegree.keys())
plt.xlabel('Node ID')
plt.ylabel('In-degree')
plt.title('In-degree distribution of tw_net_seeds')
plt.show()

# bar plot of the out-degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(tw_net_seeds_outdegree.keys())), tw_net_seeds_outdegree.values(), color='red', alpha=0.7)
plt.xticks(range(len(tw_net_seeds_outdegree.keys())), tw_net_seeds_outdegree.keys())
plt.xlabel('Node ID')
plt.ylabel('Out-degree')
plt.title('Out-degree distribution of tw_net_seeds')
plt.show()


In [ ]:
tw_net_sd.number_of_nodes(), tw_net_sd.number_of_edges()

In [ ]:
# show edges of tw_net_sd
edgelist = list(tw_net_sd.edges())


In [ ]:
# Crea un grafo vuoto
B = nx.Graph()

# Aggiungi gli archi dalla edgelist
B.add_edges_from(edgelist)

# Identifica i due insiemi:
# Assumiamo che i primi elementi delle tuple (nodi seed) appartengano a un insieme
# e i secondi elementi (nodi discovered) appartengano all'altro insieme.
seeds = {edge[0] for edge in edgelist}
discovered = {edge[1] for edge in edgelist}

# Assegna l'attributo 'bipartite' ad ogni nodo:
for nodo in seeds:
    B.nodes[nodo]['bipartite'] = 0  # Insieme 0 (ad es. Stakeholders)
for nodo in discovered:
    B.nodes[nodo]['bipartite'] = 1  # Insieme 1 (Organizzazioni o siti esterni)

In [ ]:
# Calcola la disposizione dei nodi (layout bipartito)
pos = nx.bipartite_layout(B, seeds)

# Disegna la rete
plt.figure(figsize=(8, 6))
nx.draw(B, pos, with_labels=True, node_color=['skyblue' if B.nodes[n].get('bipartite') == 0 else 'lightgreen' for n in B.nodes()], node_size=150, font_size=10)
plt.title("Rete Bipartita: Stakeholders vs Organizzazioni")
plt.show()

In [ ]:
# print the bi-adjacency matrix of the bipartite graph B
from scipy.sparse import dok_matrix

# Map original node IDs (which can be large) to consecutive row/col indices
row_nodes = sorted({u for u, _ in edgelist})
col_nodes = sorted({v for _, v in edgelist})

row_index = {n: i for i, n in enumerate(row_nodes)}
col_index = {n: j for j, n in enumerate(col_nodes)}

bi_adj_matrix = dok_matrix((len(row_nodes), len(col_nodes)), dtype=int)
for u, v in edgelist:
    bi_adj_matrix[row_index[u], col_index[v]] = 1

print(bi_adj_matrix)

In [ ]:
tw_seeds = bipartite.projected_graph(B, seeds)
tw_seeds_weighted = bipartite.weighted_projected_graph(B, seeds)

tw_discovered = bipartite.projected_graph(B, discovered)
tw_discovered_weighted = bipartite.weighted_projected_graph(B, discovered)

In [ ]:
# Visualizza la rete proiettata sui nodi seeds (non ponderata)
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(tw_seeds)  # Utilizza un layout spring per la visualizzazione
nx.draw(tw_seeds, pos, with_labels=True, node_color='skyblue', node_size=400, font_size=10)
plt.title("Proiezione sui Nodi Seeds")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(tw_seeds_weighted)
edge_widths = [tw_seeds_weighted[u][v]['weight']/10 for u, v in tw_seeds_weighted.edges()]
nx.draw(tw_seeds_weighted, pos, with_labels=True, node_color='skyblue', 
        node_size=150, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata sui Nodi Seeds")
plt.show()

In [ ]:
# plot h_net_seeds
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(tw_net_seeds)
nx.draw(tw_net_seeds, pos, node_size=350, node_color='red', edge_color='gray', with_labels=True)
plt.show()

In [ ]:
tw_net_seeds.number_of_nodes(), tw_net_seeds.number_of_edges()

In [ ]:
G_filtered.number_of_nodes(), G_filtered.number_of_edges()

In [ ]:
seeds = h_nodes[h_nodes['STATUS']=='IN']
seeds['Stance']=[0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]

In [ ]:
# Soglia desiderata (ad esempio, consideriamo solo archi con weight >= 2)
threshold = 1

# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()

# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(h_net_seeds.nodes(data=True))

# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in h_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=data['weight'])

plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color=node_colors,
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()

In [ ]:
# Soglia desiderata (ad esempio, consideriamo solo archi con weight >= 2)
threshold = 6

# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()

# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(tw_net_seeds.nodes(data=True))

# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in tw_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=data['weight'])

edges_G1 = set(tw_net_seeds.edges())
edges_G2 = set(G_filtered.edges())

intersection = edges_G1.intersection(edges_G2)
union = edges_G1.union(edges_G2)

jaccard_index = len(intersection) / len(union) if union else 0
print("Jaccard Index:", jaccard_index)


# Converte le matrici di adiacenza in array NumPy
A1 = nx.to_numpy_array(tw_net_seeds, nodelist=sorted(tw_net_seeds.nodes()))
A2 = nx.to_numpy_array(G_filtered, nodelist=sorted(G_filtered.nodes()))

# Calcola la distanza di Hamming
hamming_distance = np.sum(np.abs(A1 - A2))
print("Hamming Distance:", hamming_distance)


plt.figure(figsize=(6, 5))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color='skyblue',
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()

In [ ]:
# plot the largest connected component

plt.figure(figsize=(8, 6))
largest_cc = max(nx.connected_components(G_filtered), key=len)
G_filtered_lcc = G_filtered.subgraph(largest_cc)
pos = nx.spring_layout(G_filtered_lcc)

node_colors = []
for i, node in enumerate(G_filtered.nodes()):
    if seeds.iloc[i]['Stance'] == 1:
        node_colors.append('green')
    elif seeds.iloc[i]['Stance'] == 0:
        node_colors.append('red')

edge_widths = [G_filtered_lcc[u][v]['weight']/10 for u, v in G_filtered_lcc.edges()]
nx.draw(G_filtered_lcc, pos, with_labels=True, node_color=node_colors,
        node_size=200, font_size=10, width=edge_widths)
plt.title("Largest Connected Component")
plt.show()

In [ ]:
# remove self-loops from tw_net_seeds
tw_net_seeds.remove_edges_from(nx.selfloop_edges(tw_net_seeds))
#plot
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(tw_net_seeds)
nx.draw(tw_net_seeds, pos, node_size=250, node_color='red', edge_color='gray', with_labels=True)
plt.show()


In [ ]:
# plot h_net_seeds
plt.figure(figsize=(5, 5))
pos = nx.spring_layout(tw_net_seeds)
nx.draw(tw_net_seeds, pos, node_size=350, node_color='red', edge_color='gray', with_labels=True)
plt.show()

In [ ]:
jaccard = []
hamming = []

for threshold in range(0, 100):
    # Crea un nuovo grafo filtrato
    G_filtered = nx.Graph()

    # Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
    G_filtered.add_nodes_from(tw_net_seeds.nodes(data=True))

    # Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
    for u, v, data in tw_seeds_weighted.edges(data=True):
        if data['weight'] >= threshold:
            G_filtered.add_edge(u, v, weight=data['weight'])

    edges_G1 = set(tw_net_seeds.edges())
    edges_G2 = set(G_filtered.edges())

    intersection = edges_G1.intersection(edges_G2)
    union = edges_G1.union(edges_G2)

    jaccard_index = len(intersection) / len(union) if union else 0
    jaccard.append(jaccard_index)


    # Converte le matrici di adiacenza in array NumPy
    A1 = nx.to_numpy_array(tw_net_seeds, nodelist=sorted(tw_net_seeds.nodes()))
    A2 = nx.to_numpy_array(G_filtered, nodelist=sorted(G_filtered.nodes()))

    # Calcola la distanza di Hamming
    hamming_distance = np.sum(np.abs(A1 - A2))
    hamming.append(hamming_distance)



In [ ]:
jaccard = np.array(jaccard)
hamming = np.array(hamming)


In [ ]:
np.argmax(jaccard), np.max(jaccard)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(range(0, 100), jaccard, label="Jaccard Index")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(range(0, 100), hamming, label="Jaccard Index")
plt.show()

## Retweet

In [ ]:
# Load the data
tw_nodes = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_nodes')
tw_edges = pd.read_excel('twitter_data_def.xlsx', sheet_name='tw_edges')

In [ ]:
tw_nodes = tw_nodes.merge(h_nodes[['ID', 'STATUS']], left_on='ID', right_on='ID', how='left')

In [ ]:
RT_edges = tw_edges[tw_edges['Relationship']=='Retweet']

In [ ]:
RT_edges = RT_edges[RT_edges['Source'] != RT_edges['Target']]


In [ ]:
RT_edges.reset_index(drop=True, inplace=True)

In [ ]:
RT_edges

In [ ]:
RT_edges['Source'].unique()

In [ ]:
tw_nodes[tw_nodes['STATUS']=='IN']['ID'].unique()

In [ ]:
nodes_missing = set(tw_nodes[tw_nodes['STATUS']=='IN']['ID'].unique()) - set(RT_edges['Source'].unique())

In [ ]:
RT_net_seed = nx.DiGraph()
RT_net_bipa = nx.DiGraph()
for i in range(tw_nodes.shape[0]):
    RT_net_bipa.add_node(tw_nodes['ID'][i])
    if tw_nodes['STATUS'][i] == 'IN':
        RT_net_seed.add_node(tw_nodes['ID'][i])
for i in range(RT_edges.shape[0]):
    if RT_edges['Source'][i] in RT_net_seed.nodes() and RT_edges['Target'][i] in RT_net_seed.nodes():
        RT_net_seed.add_edge(RT_edges['Source'][i], RT_edges['Target'][i])
    else:
        RT_net_bipa.add_edge(RT_edges['Source'][i], RT_edges['Target'][i])

# remove self loops
#RT_net_seed.remove_edges_from(nx.selfloop_edges(RT_net_seed))
#RT_net_bipa.remove_edges_from(nx.selfloop_edges(RT_net_bipa))

In [ ]:
RT_net_seed.number_of_nodes(), RT_net_seed.number_of_edges()

In [ ]:
seeds_nodes = h_nodes[h_nodes['STATUS']=='IN']
seeds_nodes['Stance']=[0,0,0,1,1,0,1,0,1,1,1,0,1,1,1,1]

In [ ]:
plt.figure(figsize=(5, 5))
plt.title('Retweet Network of Seeds with Stance (LCC)')
pos = nx.spring_layout(RT_net_seed)
nx.draw(RT_net_seed, pos, node_size=350, node_color=node_colors, edge_color='gray', with_labels=True)
plt.show()

In [ ]:
node_colors = []
for i, node in enumerate(tw_net_seeds.nodes()):
    if seeds_nodes.iloc[i]['Stance'] == 1:
        node_colors.append('green')
    elif seeds_nodes.iloc[i]['Stance'] == 0:
        node_colors.append('red')
    
seeds_nodes['Color'] = node_colors  

In [ ]:
# plot RT_net_seed
# consider only the largest connected component
largest_cc = max(nx.weakly_connected_components(RT_net_seed), key=len)
RT_net_seed_lcc = RT_net_seed.subgraph(largest_cc)

# nodes removed
nodes_removed = set(RT_net_seed.nodes()) - set(RT_net_seed_lcc.nodes())
# remove nodes_removed from node_colors
node_colors_LCC = [node_colors[i] for i in range(len(node_colors)) if i not in nodes_removed]

In [ ]:
nodes_removed

In [ ]:
seeds_nodes[seeds_nodes['ID'].isin(RT_net_seed_lcc.nodes())]['Color']

In [ ]:
plt.figure(figsize=(5, 5))
plt.title('Retweet Network of Seeds with Stance (LCC)')
pos = nx.spring_layout(RT_net_seed_lcc)
nx.draw(RT_net_seed_lcc, pos, node_size=350, node_color=seeds_nodes[seeds_nodes['ID'].isin(RT_net_seed_lcc.nodes())]['Color'], edge_color='gray', with_labels=True)
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
plt.title('Retweet Network of Seeds with Stance (LCC)')
pos = nx.spring_layout(RT_net_seed_lcc)
nx.draw(RT_net_seed_lcc, pos, node_size=350, node_color=seeds_nodes[seeds_nodes['ID'].isin(RT_net_seed_lcc.nodes())]['Color'], edge_color='gray', with_labels=True)
plt.show()

In [ ]:
# compute the degree of each node in RT_net_seed
RT_net_seed_indegree = dict(RT_net_seed.in_degree())

# compute the out-degree of each node in RT_net_seed
RT_net_seed_outdegree = dict(RT_net_seed.out_degree())

# histogram of in-degrees
plt.hist(list(RT_net_seed_indegree.values()), bins=19, color='red', alpha=0.7)
plt.xlabel('In-degree')
plt.ylabel('Frequency')     
plt.title('In-degree distribution of RT_net_seed')
plt.show()

# histogram of out-degrees
plt.hist(list(RT_net_seed_outdegree.values()), bins=19, color='red', alpha=0.7)
plt.xlabel('Out-degree')
plt.ylabel('Frequency')
plt.title('Out-degree distribution of RT_net_seed')
plt.show()


In [ ]:
# bar plot of the in-degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(RT_net_seed_indegree.keys())), RT_net_seed_indegree.values(), color=node_colors, alpha=0.7)
plt.xticks(range(len(RT_net_seed_indegree.keys())), RT_net_seed_indegree.keys())
plt.xlabel('Node ID')
plt.ylabel('In-degree')
plt.title('In-degrees of RT_net_seed with Stance')
plt.show()

# bar plot of the out-degree distribution
plt.figure(figsize=(10, 5))
plt.bar(range(len(RT_net_seed_outdegree.keys())), RT_net_seed_outdegree.values(), color=node_colors, alpha=0.7)
plt.xticks(range(len(RT_net_seed_outdegree.keys())), RT_net_seed_outdegree.keys())
plt.xlabel('Node ID')
plt.ylabel('Out-degree')
plt.title('Out-degrees of RT_net_seed with Stance')
plt.show()

In [ ]:
RT_net_bipa.number_of_nodes(), RT_net_bipa.number_of_edges()

In [ ]:
# Crea un grafo vuoto
B = nx.Graph()

# Aggiungi gli archi dalla edgelist
B.add_edges_from(list(RT_net_bipa.edges()))

# Identifica i due insiemi:
# Assumiamo che i primi elementi delle tuple (nodi seed) appartengano a un insieme
# e i secondi elementi (nodi discovered) appartengano all'altro insieme.
seeds = {edge[0] for edge in RT_net_bipa.edges()}
discovered = {edge[1] for edge in RT_net_bipa.edges()}
# Assegna l'attributo 'bipartite' ad ogni nodo:
for nodo in seeds:
    B.nodes[nodo]['bipartite'] = 0  # Insieme 0 (ad es. Stakeholders)
for nodo in discovered:
    B.nodes[nodo]['bipartite'] = 1  # Insieme 1 (Organizzazioni o siti esterni)

In [ ]:
seeds

In [ ]:
# Calcola la disposizione dei nodi (layout bipartito)
pos = nx.bipartite_layout(B, seeds)

# Disegna la rete
plt.figure(figsize=(8, 6))
nx.draw(B, pos, with_labels=True, node_color=['skyblue' if B.nodes[n].get('bipartite') == 0 else 'lightgreen' for n in B.nodes()], node_size=150, font_size=10)
plt.title("Rete Bipartita: Stakeholders vs Organizzazioni")
plt.show()

In [ ]:
nodes_missing

In [ ]:
node_colors_missing

In [ ]:
# Visualizza la rete proiettata sui nodi seeds (non ponderata)
tw_seeds = bipartite.projected_graph(B, seeds)
tw_seeds_weighted = bipartite.weighted_projected_graph(B, seeds)

tw_discovered = bipartite.projected_graph(B, discovered)
tw_discovered_weighted = bipartite.weighted_projected_graph(B, discovered)

node_colors_missing = [node_colors[i] for i in range(len(node_colors)) if i not in nodes_missing]


plt.figure(figsize=(8, 6))
pos = nx.spring_layout(tw_seeds_weighted)
edge_widths = [tw_seeds_weighted[u][v]['weight']/10 for u, v in tw_seeds_weighted.edges()]
nx.draw(tw_seeds_weighted, pos, with_labels=True, node_color=seeds_nodes[seeds_nodes['ID'].isin(tw_seeds.nodes())]['Color'], 
        node_size=150, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata sui Nodi Seeds")
plt.show()


threshold = 3
# Crea un nuovo grafo filtrato
G_filtered = nx.Graph()
# Aggiungi i nodi dal grafo originale (assicurati di mantenere le informazioni necessarie)
G_filtered.add_nodes_from(RT_net_seed.nodes(data=True))
# Itera sugli archi e aggiungi solo quelli che soddisfano la soglia
for u, v, data in tw_seeds_weighted.edges(data=True):
    if data['weight'] >= threshold:
        G_filtered.add_edge(u, v, weight=1)
# lcc
G_filtered = G_filtered.subgraph(max(nx.connected_components(G_filtered), key=len))

plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G_filtered)  # Layout per la visualizzazione
edge_widths = [G_filtered[u][v]['weight'] for u, v in G_filtered.edges()]
nx.draw(G_filtered, pos, with_labels=True, node_color=seeds_nodes[seeds_nodes['ID'].isin(G_filtered.nodes())]['Color'],
        node_size=200, font_size=10, width=edge_widths)
plt.title("Proiezione Ponderata Filtrata (threshold >= {})".format(threshold))
plt.show()


plt.figure(figsize=(8, 6))
pos = nx.spring_layout(tw_seeds)  # Utilizza un layout spring per la visualizzazione
nx.draw(tw_seeds, pos, with_labels=True, node_color=seeds_nodes[seeds_nodes['ID'].isin(tw_seeds.nodes())]['Color'], node_size=400, font_size=10)
plt.title("Proiezione sui Nodi Seeds")
plt.show()


